In [1]:
!pip install -q torch transformers ninja tokenizers rwkv pynvml huggingface_hub smolagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.0/410.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [2]:
# tools
!pip install -q ddgs wikipedia-api

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 81.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.9 MB/s eta 0:00:00:00:01


In [3]:
import os
import re
import copy
import time
import gc

import torch
import huggingface_hub
from transformers import AutoTokenizer

from smolagents.agents import EMPTY_PROMPT_TEMPLATES
from smolagents.models import Model, ChatMessage, MessageRole
from smolagents.monitoring import TokenUsage
from smolagents.tools import Tool
from smolagents.default_tools import (
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    VisitWebpageTool,
    WikipediaSearchTool
)
from smolagents import CodeAgent

from IPython.display import clear_output

In [4]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

from rwkv.model import RWKV
from rwkv.utils import PIPELINE, PIPELINE_ARGS

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv_cuda/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] c++ -MMD -MF gemm_fp16_cublas.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/gemm_fp16_cublas.cpp -o gemm_fp16_cublas.o 
[2/4] c++ -MMD -MF wrapper.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-p

Loading extension module wkv_cuda...
Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv7s...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv7s/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv7s...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF rwkv7_op.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/rwkv7_op.cpp -o rwkv7_op.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output rwkv7.cuda.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include

Loading extension module wkv7s...


In [5]:
model_title = model_title = "rwkv7-g1a-2.9b-20250924-ctx4096"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")
rwkv_model = RWKV(model=model_path.replace('.pth',''), strategy='cuda fp16')
rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")

rwkv7-g1a-2.9b-20250924-ctx4096.pth:   0%|          | 0.00/5.90G [00:00<?, ?B/s]

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/71df75d4c929166e220998f32bdf3139e54c36b8/rwkv7-g1a-2.9b-20250924-ctx4096 (cuda fp16)



In [49]:
class RWKVModel(Model):
    """
    A subclass of Model for RWKV7 inference, adapted from the experimental notebook.
    """
    def __init__(
        self,
        model,
        pipeline,
        flatten_messages_as_text: bool = False,
        tool_name_key: str = "name",
        tool_arguments_key: str = "arguments",
        model_id: str | None = None,
        **kwargs,
    ):    
        # Setup RWKV
        self.CTX_LIMIT = 3000
        self.PENALTY_DECAY = 0.996
        self._model = model
        self._pipeline = pipeline

        # Store state of the model
        self.state = None
        
        # Only for chat template
        # TODO: Get rid of it...
        self.tokenizer = AutoTokenizer.from_pretrained('fla-hub/rwkv7-2.9B-g1', trust_remote_code=True)
        
        super().__init__(flatten_messages_as_text=True, model_id="model_id", **kwargs)

    def reset_state(self) -> None:
        self.state = None
    
    def generate(
        self,
        messages: list[ChatMessage],
        stop_sequences: list[str] | None = None,
        response_format: dict[str, str] | None = None,
        tools_to_call_from: list[Tool] | None = None,
        **kwargs,
    ) -> ChatMessage:
        """Process the input messages and return the model's response."""
        if kwargs.get("reset_state", False):
            print("Reseting state...")
            self.reset_state()
        else:
            print("Reuse previous state...")
        
        completion_kwargs = self._prepare_completion_kwargs(
            messages=messages,
            flatten_messages_as_text=self.flatten_messages_as_text,
            stop_sequences=stop_sequences,
            tools_to_call_from=tools_to_call_from,
            **kwargs,
        )

        print("Completion kwargs", completion_kwargs)

        messages = self.prepare_messages(completion_kwargs.pop("messages"))
        prepared_stop_sequences = completion_kwargs.pop("stop", [])
        tools = completion_kwargs.pop("tools", None)
        completion_kwargs.pop("tool_choice", None)

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            tools=tools,
            add_generation_prompt=True,
            enable_thinking=False
        )

        # Remove <|rwkv_tokenizer_end_of_text|>
        prompt = prompt.replace("<|rwkv_tokenizer_end_of_text|>", "").strip()

        print(f"Input prompt: ", prompt)

        output = self.run_rwkv(prompt)
        # Test string
        # output = """Thought: I will use the `math` module to compute the value of exp(10) and then add it to 2.\n<code>\nimport math\nresult = 2 + math.exp(10)\nprint(result)\n</code>"""

        output_text = output.strip()

        print(f"Output: ", output_text)
        
        return ChatMessage(
            role=MessageRole.ASSISTANT,
            content=output_text,
            raw={"out": output_text},
            token_usage=TokenUsage(
                input_tokens=len(prompt.split()),
                output_tokens=len(output_text.split()),
            ),
        )

    def run_rwkv(
        self,
        ctx,
        token_count=500,
        temperature=1.0,
        top_p=0.7,
        presencePenalty=0.1,
        countPenalty=0.1,
    ):
        # TODO: Use generate(), or get rid of PIPELINE_ARGS
        args = PIPELINE_ARGS(
            temperature=max(0.2, float(temperature)),
            top_p=float(top_p),
            alpha_frequency=countPenalty,
            alpha_presence=presencePenalty,
            token_ban=[],        # ban the generation of some tokens
            token_stop=[0]       # stop generation whenever you see any token here
        )
    
        ctx = ctx.strip()
        all_tokens = []
        out_last = 0
        out_str = ''
        occurrence = {}
    
        for i in range(int(token_count)):
            input_ids = self._pipeline.encode(ctx)
            input_ids = input_ids[-self.CTX_LIMIT:] if i == 0 else [token]
            
            out, self.state = self._model.forward(input_ids, self.state)
    
            for n in occurrence:
                out[n] -= (args.alpha_presence + occurrence[n] * args.alpha_frequency)
    
            token = self._pipeline.sample_logits(
                out,
                temperature=args.temperature,
                top_p=args.top_p)
            if token in args.token_stop:
                break
    
            all_tokens.append(token)
    
            for xxx in occurrence:
                occurrence[xxx] *= self.PENALTY_DECAY
    
            ttt = self._pipeline.decode([token])
            www = 1
            if ttt in ' \t0123456789':
                www = 0
            if token not in occurrence:
                occurrence[token] = www
            else:
                occurrence[token] += www
    
            tmp = self._pipeline.decode(all_tokens[out_last:])
            if '\ufffd' not in tmp:
                out_str += tmp
                out_last = i + 1

                if out_str.strip().endswith("</code>"):
                    break
    
                if out_str.strip().endswith("Observation:"):
                    break
    
        # Cleanup and timing
        del out
        gc.collect()
        torch.cuda.empty_cache()
        
        return out_str.strip()

    def prepare_messages(self, messages: list[dict[str]]) -> list[dict[str]]:
        new_messages = []
    
        for m in messages:
            content = m.get("content", "")
            if content.strip():
                content = re.sub(r'\n+', '\n', content)
                content = re.sub(r'\n+$', '', content)
                m["content"] = content
                new_messages.append(m)

        return new_messages

In [50]:
model = RWKVModel(model=rwkv_model, pipeline=rwkv_pipeline)

In [51]:
model.run_rwkv("User: Continue previous sentence.\n\nAssistant:<think>\n</think>")

"The issue with the above code is that the user is trying to print the value of `a` (which is `5`) instead of the variable `num` (which holds the value `5`). In Python, when you want to print a variable's value, you should use the variable name, not the value itself. The correct code should be:\n```python\nnum = 5\nprint(num)\n```\nHowever, since the user's original code already has `print(num)`, which is correct, the error might be due to a misunderstanding of the code's purpose. The code is simply printing the value of `num`, which is `5`. There is no syntax error in the code as written."

In [52]:
model.reset_state()

In [68]:
SYSTEM_PROMPT = '''You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '{{code_block_opening_tag}}', and closed with '{{code_block_closing_tag}}'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.
Here are a few examples using notional tools:
---
User: New task:
What is the result of the following operation: 5 + 3 + 1294.678?

Assistant:<think>
</think>Thought: I will use Python code to compute the result of the operation and then return the final answer using the `final_answer` tool.
{{code_block_opening_tag}}
result = 5 + 3 + 1294.678
final_answer(result)
{{code_block_closing_tag}}
---
User: New task:
"What is the current age of the pope, raised to the power 0.36?"

Assistant:<think>
</think>Thought: I will use the tool `wikipedia_search` to get the age of the pope, and confirm that with a web search.
{{code_block_opening_tag}}
pope_age_wiki = wikipedia_search(query="current pope age")
print("Pope age as per wikipedia:", pope_age_wiki)
pope_age_search = web_search(query="current pope age")
print("Pope age as per google search:", pope_age_search)
{{code_block_closing_tag}}
Observation:
Pope age: "The pope Francis is currently 88 years old."
Thought: I know that the pope is 88 years old. Let's compute the result using Python code.
{{code_block_opening_tag}}
pope_current_age = 88 ** 0.36
final_answer(pope_current_age)
{{code_block_closing_tag}}
---
Above examples were using notional tools that might not exist for you.
On top of performing computations in the Python code snippets that you create, you only have access to these tools, behaving like regular python functions:
{{code_block_opening_tag}}
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{%- endfor %}
{{code_block_closing_tag}}
Here are the rules you should always follow to solve your task:
1. Always provide a 'Thought:' sequence, and a '{{code_block_opening_tag}}' sequence ending with '{{code_block_closing_tag}}', else you will fail.
2. Use only variables that you have defined!
3. Use only tools that provided to you!
4. To complete the task you need to call ```final_answer()``` tool!
5. Always use the right arguments for the tools. DO NOT pass the arguments as a dict as in 'answer = wikipedia_search({'query': "What is the place where James Bond lives?"})', but use the arguments directly as in 'answer = wikipedia_search(query="What is the place where James Bond lives?")'.
6. Call a tool only when needed, and never re-do a tool call that you previously did with the exact same parameters.
7. Don't name any new variable with the same name as a tool: for instance don't name a variable 'final_answer'.
8. Never create any notional variables in our code, as having these in your logs will derail you from the true variables.
9. You can use imports in your code, but only from the following list of modules: {{authorized_imports}}
10. The state persists between code executions: so if in one step you've created variables or imported modules, these will all persist.
11. Don't give up! You're in charge of solving the task, not providing directions to solve it.
Now Begin!'''

In [69]:
RWKV_PROMPT_TEMPLATES = EMPTY_PROMPT_TEMPLATES
RWKV_PROMPT_TEMPLATES["system_prompt"] = SYSTEM_PROMPT

In [70]:
agent = CodeAgent(
    tools=[FinalAnswerTool(), DuckDuckGoSearchTool(), VisitWebpageTool(), WikipediaSearchTool()],
    model=model,
    prompt_templates=RWKV_PROMPT_TEMPLATES,
    verbosity_level=2)

In [71]:
model.reset_state()
agent.run("What is the result of 2 + exp(10)?", max_steps=2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the result of 2 + exp(10)?                                                                              │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reuse previous state...
Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': 'You are an expert assistant who can solve any task using code blobs.\nYou will be given a task to solve as best you can.\nTo do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.\nTo solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.\nAt each step, in the \'Thought:\' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.\nThen in the Code sequence you should write the code in simple Python. The code sequence must be opened with \'<code>\', and closed with \'</code>\'.\nDuring each intermediate step, you can use \'print()\' to save whatever important information you will then need.\nThese print outputs will then appear in the \'Observation:\' field, which will be available as i

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will use Python code to compute the result of the operation and then return the final answer using the  
`final_answer` tool.                                                                                               
<code>                                                                                                             
result = 2 + exp(10)                                                                                               
final_answer(result)                                                                                               
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = 2 + exp(10)                                                                                             
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 22028.465794806718

[Step 1: Duration 3.54 seconds| Input tokens: 745 | Output tokens: 31]

22028.465794806718

In [72]:
model.reset_state()
agent.run("What is the most popular programming language? Output single name.", max_steps=1)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the most popular programming language? Output single name.                                              │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reuse previous state...
Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': 'You are an expert assistant who can solve any task using code blobs.\nYou will be given a task to solve as best you can.\nTo do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.\nTo solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.\nAt each step, in the \'Thought:\' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.\nThen in the Code sequence you should write the code in simple Python. The code sequence must be opened with \'<code>\', and closed with \'</code>\'.\nDuring each intermediate step, you can use \'print()\' to save whatever important information you will then need.\nThese print outputs will then appear in the \'Observation:\' field, which will be available as i

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to determine the most popular programming language using reliable data sources. The most reliable  
source is usually the TIOBE Index, which ranks programming languages based on search engine results.               
<code>                                                                                                             
tiobe_index = web_search("most popular programming language TIOBE Index")                                          
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tiobe_index = web_search("most popular programming language TIOBE Index")                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

[Step 1: Duration 5.14 seconds| Input tokens: 747 | Output tokens: 42]

Reuse previous state...
Completion kwargs {'messages': [{'role': <MessageRole.SYSTEM: 'system'>, 'content': ''}, {'role': <MessageRole.USER: 'user'>, 'content': 'New task:\nWhat is the most popular programming language? Output single name.'}, {'role': <MessageRole.ASSISTANT: 'assistant'>, 'content': 'Thought: I need to determine the most popular programming language using reliable data sources. The most reliable source is usually the TIOBE Index, which ranks programming languages based on search engine results.\n<code>\ntiobe_index = web_search("most popular programming language TIOBE Index")\n</code>\nCalling tools:\n[{\'id\': \'call_1\', \'type\': \'function\', \'function\': {\'name\': \'python_interpreter\', \'arguments\': \'tiobe_index = web_search("most popular programming language TIOBE Index")\'}}]'}, {'role': <MessageRole.USER: 'user'>, 'content': 'Observation:\nExecution logs:\nLast output from code snippet:\n## Search Results\n\n[TIOBE Index - TIOBE](https://www.tiobe.com/tio

Reached max steps.

[Step 2: Duration 3.66 seconds| Input tokens: 1,360 | Output tokens: 83]

'Thought: I have found a reliable source that indicates Python is the most popular programming language. I will now confirm this with another search using a different search engine to ensure consistency.\n<code>\npython_popularity = web_search("most popular programming language Python")\n</code>'